# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup

# install piper-sample-generator PINNED to v2.0.0 -- the default branch was restructured
# (v3.x) and no longer has the generate_samples.py file train.py imports directly.
!git clone --branch v2.0.0 --depth 1 https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# clone openwakeword's source (shared by both environments below -- the editable installs
# in each just point back at this same directory, so onnx model downloads at the bottom
# only need to happen once)
!git clone https://github.com/dscripka/openwakeword

# Global Colab env only needs `datasets` -- the RIR/AudioSet download cells below call
# datasets.load_dataset() directly in this kernel, not through a subprocess.
!pip install -q datasets==2.14.6

# --- piper-phonemize==1.1.0 (needed by generate_samples.py for TTS, and transitively by
# train.py's adversarial-phrase generation) has no Linux wheel for Python 3.12, which is
# Colab's default interpreter. It DOES publish cp310/cp311 Linux wheels, so set up an
# isolated Python 3.10 venv and run every train.py invocation through it instead.
!sudo apt-get update -qq
!sudo apt-get install -y -qq software-properties-common
!sudo add-apt-repository -y ppa:deadsnakes/ppa >/dev/null 2>&1
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.10 python3.10-venv python3.10-dev
!python3.10 -m venv /content/pg_venv

# Match Colab's CUDA build (cu128) so the T4 GPU is still used inside the venv.
!/content/pg_venv/bin/pip install -q --upgrade pip
!/content/pg_venv/bin/pip install -q torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!/content/pg_venv/bin/pip install -q 'numpy<2' scipy tqdm pyyaml requests
!/content/pg_venv/bin/pip install -q piper-phonemize==1.1.0 webrtcvad
!/content/pg_venv/bin/pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations>=0.12.0 acoustics==0.2.6 pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19
!/content/pg_venv/bin/pip install -q -e ./openwakeword

# Download required onnx feature-extraction models (workaround for Colab) -- onnx only,
# we never use the .tflite versions since Sorena is pinned to the onnx runtime. Both
# environments' editable installs point at this same source directory.
import os

os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx

**Required manual step:** the cell above pins `datasets==2.14.6`, but Colab's kernel
already had a newer `datasets` imported in memory (which decodes audio into a
`torchcodec.decoders.AudioDecoder` object instead of the old `{'path','array'}` dict
this notebook's code expects) -- pip installing a different version doesn't undo an
already-imported module. Run the cell below to force a kernel restart so the pinned
version actually takes effect, wait ~10 seconds for Colab to reconnect, then use
**Runtime -> Run after** (right-click this cell, or the Runtime menu) to continue with
the rest of the notebook -- do NOT re-run the environment setup cell above.

In [ ]:
import os

os.kill(os.getpid(), 9)

In [ ]:
# Imports

import os
from pathlib import Path

import datasets
import numpy as np
import scipy
import yaml
from tqdm import tqdm

# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset(
    "davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True
)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row["audio"]["path"].split("/")[-1]
    scipy.io.wavfile.write(
        os.path.join(output_dir, name), 16000, (row["audio"]["array"] * 32767).astype(np.int16)
    )

In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict(
    {"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]}
)
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row["audio"]["path"].split("/")[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(
        os.path.join(output_dir, name), 16000, (row["audio"]["array"] * 32767).astype(np.int16)
    )

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(
    range(n_hours * 3600 // 30)
):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row["audio"]["path"].split("/")[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(
        os.path.join(output_dir, name), 16000, (row["audio"]["array"] * 32767).astype(np.int16)
    )
    i += 1
    if i == n_hours * 3600 // 30:
        break

In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version

config["target_phrase"] = ["hey sorena"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = [
    "./audioset_16k",
    "./fma",
]  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {
    "ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
}

with open("my_model.yaml", "w") as f:
    yaml.dump(config, f)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
%%bash
cat > /content/patch_and_run.py << 'PYEOF'
# Wrapper that patches torch.load BEFORE running train.py, instead of relying on
# Python's sitecustomize.py auto-import -- confirmed via direct diagnostic that this
# venv's Python does not actually auto-import sitecustomize.py at startup (a quirk of
# this particular deadsnakes/Ubuntu build), so this is a self-contained alternative
# that doesn't depend on that mechanism working at all.
import torch

_orig_load = torch.load


def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)


torch.load = _patched_load

import runpy
import sys

sys.argv = ["train.py", "--training_config", "my_model.yaml", "--generate_clips"]
runpy.run_path("openwakeword/openwakeword/train.py", run_name="__main__")
PYEOF

MPLBACKEND=Agg /content/pg_venv/bin/python /content/patch_and_run.py


In [ ]:
%%bash
cat > /content/patch_and_run.py << 'PYEOF'
# Wrapper that patches torch.load BEFORE running train.py, instead of relying on
# Python's sitecustomize.py auto-import -- confirmed via direct diagnostic that this
# venv's Python does not actually auto-import sitecustomize.py at startup (a quirk of
# this particular deadsnakes/Ubuntu build), so this is a self-contained alternative
# that doesn't depend on that mechanism working at all.
import torch

_orig_load = torch.load


def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)


torch.load = _patched_load

import runpy
import sys

sys.argv = ["train.py", "--training_config", "my_model.yaml", "--augment_clips"]
runpy.run_path("openwakeword/openwakeword/train.py", run_name="__main__")
PYEOF

MPLBACKEND=Agg /content/pg_venv/bin/python /content/patch_and_run.py


In [ ]:
%%bash
cat > /content/patch_and_run.py << 'PYEOF'
# Wrapper that patches torch.load BEFORE running train.py, instead of relying on
# Python's sitecustomize.py auto-import -- confirmed via direct diagnostic that this
# venv's Python does not actually auto-import sitecustomize.py at startup (a quirk of
# this particular deadsnakes/Ubuntu build), so this is a self-contained alternative
# that doesn't depend on that mechanism working at all.
import torch

_orig_load = torch.load


def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)


torch.load = _patched_load

import runpy
import sys

sys.argv = ["train.py", "--training_config", "my_model.yaml", "--train_model"]
runpy.run_path("openwakeword/openwakeword/train.py", run_name="__main__")
PYEOF

MPLBACKEND=Agg /content/pg_venv/bin/python /content/patch_and_run.py


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!

In [ ]:
# Download the trained model to your computer
from google.colab import files

files.download(f"my_custom_model/{config['model_name']}.onnx")